# 03 - Integración de datos

Une la meteorología agregada por CCAA (`01_limpieza_aemet`) con los pesos de potencia fotovoltaica (`02_limpieza_ree`) para obtener la meteorología nacional ponderada, y la combina con la generación fotovoltaica nacional diaria para formar la primera versión de la capa Gold: una fila por día.

## 1. Carga de las salidas de los notebooks anteriores

In [38]:
import pandas as pd
import numpy as np

media_ccaa_dia = pd.read_parquet(
    "../data/processed/aemet_media_ccaa_diaria.parquet"
)

pesos = pd.read_parquet(
    "../data/processed/pesos_ccaa_por_año.parquet"
)

ree = pd.read_csv(
    "../data/processed/ree_fotovoltaica_diario_limpio_2021_2025.csv",
    parse_dates=["fecha"]
)

print("media_ccaa_dia:", media_ccaa_dia.shape)
print("pesos:", pesos.shape)
print("ree:", ree.shape)

media_ccaa_dia: (34694, 12)
pesos: (95, 3)
ree: (1825, 3)


In [39]:
media_ccaa_dia.head()

,fecha,ccaa,tmed,tmin,tmax,prec,velmedia,racha,hrMedia,presMax,presMin,sol
0,2021-01-01,Andalucia,7.754464,4.417857,11.099107,0.382727,4.078571,12.128866,67.778761,979.944737,976.839474,5.053846
1,2021-01-01,Aragon,2.174138,-1.320690,5.662069,0.242105,3.398214,10.560714,76.245614,935.409091,933.100000,5.436364
2,2021-01-01,Asturias,2.724000,0.808000,4.644000,12.655000,2.645455,11.914286,90.920000,968.388889,963.500000,0.671429
3,2021-01-01,Baleares,8.787179,5.285000,12.253846,15.497500,2.761290,13.426667,83.179487,1002.716667,995.783333,1.860000
4,2021-01-01,Canarias,14.648649,12.179730,17.133784,0.333784,3.595775,10.078571,66.891892,986.611111,983.850000,2.853333


In [40]:
pesos.head()

,año,ccaa,peso
0,2021,Extremadura,0.235391
1,2021,Andalucia,0.234290
2,2021,Castilla la Mancha,0.178602
3,2021,Murcia,0.095776
4,2021,Aragon,0.091871


In [41]:
ree.head()

,generacion_fotovoltaica,porcentaje_fotovoltaica,fecha
0,34247.163,0.021989,2021-01-02
1,37134.485,0.024855,2021-01-03
2,26307.981,0.017347,2021-01-04
3,29746.352,0.020739,2021-01-05
4,25273.930,0.018880,2021-01-06


## 2. Validación de nombres de CCAA antes del merge

Punto crítico: si los nombres de `ccaa` no coinciden carácter a carácter entre las dos tablas, el merge deja esas filas con `peso = NaN` sin lanzar ningún error visible.

In [42]:
ccaa_excluidas = {"Melilla"}

ccaa_aemet = set(media_ccaa_dia["ccaa"].unique()) - ccaa_excluidas
ccaa_pesos = set(pesos["ccaa"].unique())

solo_en_aemet = ccaa_aemet - ccaa_pesos
solo_en_pesos = ccaa_pesos - ccaa_aemet

print("CCAA solo en AEMET (sin peso asociado):", solo_en_aemet)
print("CCAA solo en pesos (no aparecen en AEMET):", solo_en_pesos)

assert len(solo_en_aemet) == 0, (
    f"Hay CCAA en la meteorologia sin peso asociado: {solo_en_aemet}. "
    "Revisa la normalizacion de nombres en 01_limpieza_aemet y 02_limpieza_ree."
)

media_ccaa_dia = media_ccaa_dia[~media_ccaa_dia["ccaa"].isin(ccaa_excluidas)].copy()

CCAA solo en AEMET (sin peso asociado): set()
CCAA solo en pesos (no aparecen en AEMET): {'Melilla'}


## 3. Paso 2 de la agregación espacial: media ponderada nacional

In [43]:
media_ccaa_dia["año"] = media_ccaa_dia["fecha"].dt.year

meteo_ccaa_pesos = media_ccaa_dia.merge(
    pesos,
    on=["ccaa", "año"],
    how="left"
)

filas_faltantes = meteo_ccaa_pesos[meteo_ccaa_pesos["peso"].isna()]

print("CCAA y años sin peso asociado:")
print(
    filas_faltantes
    .groupby(["ccaa", "año"])
    .size()
    .reset_index(name="dias_afectados")
)

# Validacion: ninguna fila deberia quedarse sin peso tras el merge
filas_sin_peso = meteo_ccaa_pesos["peso"].isna().sum()
print("Filas sin peso asignado tras el merge:", filas_sin_peso)
assert filas_sin_peso == 0, "Hay filas sin peso asignado; revisa cobertura de años en pesos"

CCAA y años sin peso asociado:
Empty DataFrame
Columns: [ccaa, año, dias_afectados]
Index: []
Filas sin peso asignado tras el merge: 0


In [44]:
variables_meteo = [
    "tmed", "tmin", "tmax", "prec",
    "velmedia", "racha",
    "hrMedia", "presMax", "presMin",
    "sol"
]

def media_ponderada(g):
    resultado = {}
    for var in variables_meteo:
        # Solo se ponderan filas donde la variable no es nula ese dia,
        # para no penalizar el resultado si una CCAA falta puntualmente
        mask = g[var].notna()
        if mask.sum() == 0:
            resultado[var] = np.nan
        else:
            resultado[var] = (g.loc[mask, var] * g.loc[mask, "peso"]).sum() / g.loc[mask, "peso"].sum()
    return pd.Series(resultado)

meteo_nacional_diaria = (
    meteo_ccaa_pesos.groupby("fecha")
    .apply(media_ponderada)
    .reset_index()
)

print(meteo_nacional_diaria.shape)
meteo_nacional_diaria.head()

(1826, 11)


,fecha,tmed,tmin,tmax,prec,velmedia,racha,hrMedia,presMax,presMin,sol
0,2021-01-01,5.029027,1.524033,8.541629,0.774520,3.631369,11.812171,72.808888,958.779498,955.714960,5.469856
1,2021-01-02,3.371410,-0.635081,7.377859,0.243681,3.248500,10.651848,66.304925,961.451484,957.727905,6.313311
2,2021-01-03,3.410448,-1.765693,8.584577,0.232906,2.664880,9.554461,64.446681,962.750448,959.676937,7.293419
3,2021-01-04,3.357818,-1.088909,7.805587,0.434375,2.175468,8.057505,73.948426,959.948097,956.382750,4.613515
4,2021-01-05,2.951290,-1.842792,7.745667,0.328913,1.645162,6.131043,74.186674,958.377786,955.568340,5.315905


In [ ]:

print("% de nulos por variable tras la ponderacion nacional:")
print(meteo_nacional_diaria[variables_meteo].isna().mean().mul(100).round(2))


rango_completo = pd.date_range(
    meteo_nacional_diaria["fecha"].min(),
    meteo_nacional_diaria["fecha"].max(),
    freq="D"
)
dias_faltantes = rango_completo.difference(meteo_nacional_diaria["fecha"])
print("\nDias faltantes en la meteorologia nacional:", len(dias_faltantes))
if len(dias_faltantes) > 0:
    print(dias_faltantes)

% de nulos por variable tras la ponderacion nacional:
tmed        0.0
tmin        0.0
tmax        0.0
prec        0.0
velmedia    0.0
racha       0.0
hrMedia     0.0
presMax     0.0
presMin     0.0
sol         0.0
dtype: float64

Dias faltantes en la meteorologia nacional: 0


## 4. Unión con la generación fotovoltaica nacional

In [46]:
gold = ree.merge(
    meteo_nacional_diaria,
    on="fecha",
    how="inner"
)

print("Filas ree:", ree.shape[0])
print("Filas meteo:", meteo_nacional_diaria.shape[0])
print("Filas tras el merge:", gold.shape[0])

# Si el numero de filas tras el merge es menor que el minimo de las dos tablas,
# hay fechas que no casan entre ambas fuentes
print("\nFechas de ree sin meteorologia asociada:",
      len(set(ree["fecha"]) - set(meteo_nacional_diaria["fecha"])))
print("Fechas de meteorologia sin generacion asociada:",
      len(set(meteo_nacional_diaria["fecha"]) - set(ree["fecha"])))

Filas ree: 1825
Filas meteo: 1826
Filas tras el merge: 1825

Fechas de ree sin meteorologia asociada: 0
Fechas de meteorologia sin generacion asociada: 1


In [47]:
gold = gold.sort_values("fecha").reset_index(drop=True)
gold.head()

,generacion_fotovoltaica,porcentaje_fotovoltaica,fecha,tmed,tmin,tmax,prec,velmedia,racha,hrMedia,presMax,presMin,sol
0,34247.163,0.021989,2021-01-02,3.371410,-0.635081,7.377859,0.243681,3.248500,10.651848,66.304925,961.451484,957.727905,6.313311
1,37134.485,0.024855,2021-01-03,3.410448,-1.765693,8.584577,0.232906,2.664880,9.554461,64.446681,962.750448,959.676937,7.293419
2,26307.981,0.017347,2021-01-04,3.357818,-1.088909,7.805587,0.434375,2.175468,8.057505,73.948426,959.948097,956.382750,4.613515
3,29746.352,0.020739,2021-01-05,2.951290,-1.842792,7.745667,0.328913,1.645162,6.131043,74.186674,958.377786,955.568340,5.315905
4,25273.930,0.018880,2021-01-06,2.799222,-2.215370,7.812312,3.824368,1.976831,7.214517,71.192401,959.602209,955.569734,4.533651


## 5. Variables temporales derivadas

In [48]:
gold["dia_semana"] = gold["fecha"].dt.dayofweek
gold["mes"] = gold["fecha"].dt.month
gold["dia_año"] = gold["fecha"].dt.dayofyear

gold[["fecha", "dia_semana", "mes", "dia_año"]].head()

,fecha,dia_semana,mes,dia_año
0,2021-01-02,5,1,2
1,2021-01-03,6,1,3
2,2021-01-04,0,1,4
3,2021-01-05,1,1,5
4,2021-01-06,2,1,6


## 6. Validaciones finales antes de guardar

Comprobaciones de cierre para asegurar que la capa Gold no arrastra ningún problema silencioso de los pasos anteriores.

In [49]:
print("Filas:", gold.shape[0])
print("Columnas:", gold.shape[1])
print("Rango de fechas:", gold["fecha"].min(), "-", gold["fecha"].max())
print("Fechas duplicadas:", gold["fecha"].duplicated().sum())

print("\nNulos por columna:")
print(gold.isna().mean().mul(100).round(2).sort_values(ascending=False))

Filas: 1825
Columnas: 16
Rango de fechas: 2021-01-02 00:00:00 - 2025-12-31 00:00:00
Fechas duplicadas: 0

Nulos por columna:
generacion_fotovoltaica    0.0
porcentaje_fotovoltaica    0.0
fecha                      0.0
tmed                       0.0
tmin                       0.0
tmax                       0.0
prec                       0.0
velmedia                   0.0
racha                      0.0
hrMedia                    0.0
presMax                    0.0
presMin                    0.0
sol                        0.0
dia_semana                 0.0
mes                        0.0
dia_año                    0.0
dtype: float64


In [50]:
assert gold["fecha"].duplicated().sum() == 0, "Hay fechas duplicadas en la capa Gold"
assert gold["generacion_fotovoltaica"].isna().sum() == 0, "Hay nulos en la variable objetivo"

print("OK: validaciones superadas")

OK: validaciones superadas


## 7. Guardado de la capa Gold

In [51]:
gold.to_parquet(
    "../data/gold/gold_diario_2021_2025.parquet",
    index=False
)

gold.to_csv(
    "../data/gold/gold_diario_2021_2025.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Guardado:", gold.shape)
gold.head()

Guardado: (1825, 16)


,generacion_fotovoltaica,porcentaje_fotovoltaica,fecha,tmed,tmin,tmax,prec,velmedia,racha,hrMedia,presMax,presMin,sol,dia_semana,mes,dia_año
0,34247.163,0.021989,2021-01-02,3.371410,-0.635081,7.377859,0.243681,3.248500,10.651848,66.304925,961.451484,957.727905,6.313311,5,1,2
1,37134.485,0.024855,2021-01-03,3.410448,-1.765693,8.584577,0.232906,2.664880,9.554461,64.446681,962.750448,959.676937,7.293419,6,1,3
2,26307.981,0.017347,2021-01-04,3.357818,-1.088909,7.805587,0.434375,2.175468,8.057505,73.948426,959.948097,956.382750,4.613515,0,1,4
3,29746.352,0.020739,2021-01-05,2.951290,-1.842792,7.745667,0.328913,1.645162,6.131043,74.186674,958.377786,955.568340,5.315905,1,1,5
4,25273.930,0.018880,2021-01-06,2.799222,-2.215370,7.812312,3.824368,1.976831,7.214517,71.192401,959.602209,955.569734,4.533651,2,1,6
